The dataset is stored in a database to enable faster access and management of large data. Python is then used to run SQL queries such as SELECT, JOIN, and INSERT to explore the data and generate insights before performing EDA and statistical analysis.

In [4]:
import pandas as pd
from sqlalchemy import create_engine, text
import sqlite3

In [6]:
main_df = pd.read_csv("/Users/utkuseyithanoglu/Desktop/nyc car crash analysis/utku-explo/clean_crash_data.csv")
print(f"Loaded {len(main_df)}rows from CSV")
print(f"Columns: {main_df.columns.tolist()}")

Loaded 2036397rows from CSV
Columns: ['CRASH DATE', 'CRASH TIME', 'BOROUGH', 'ZIP CODE', 'LATITUDE', 'LONGITUDE', 'NUMBER OF PERSONS INJURED', 'NUMBER OF PERSONS KILLED', 'NUMBER OF PEDESTRIANS INJURED', 'NUMBER OF PEDESTRIANS KILLED', 'NUMBER OF CYCLIST INJURED', 'NUMBER OF CYCLIST KILLED', 'NUMBER OF MOTORIST INJURED', 'NUMBER OF MOTORIST KILLED', 'CONTRIBUTING FACTOR VEHICLE 1', 'COLLISION_ID', 'day_of_week', 'is_weekend', 'hour', 'injury_flag', 'vehicle_category']


In [9]:
engine = create_engine('sqlite:////Users/utkuseyithanoglu/Desktop/nyc car crash analysis/utku-explo/clean_crash_data.db')

In [10]:
main_df.to_sql('nyc_car_crash',engine, if_exists='replace',index=False)
print("✓ Data successfully stored in SQLite database!")

✓ Data successfully stored in SQLite database!


The cleaned dataset was loaded from a CSV file and stored in a SQLite database to enable faster access and querying before EDA.

In [11]:
## example sql queries


In [20]:
with engine.connect() as conn:
    # According to the Borough, the number of accidents
    result = conn.execute(text("""
    SELECT *
    FROM nyc_car_crash
    LIMIT 2;"""))
    for i in result:
        print(i)


('2023-11-01', '1:29', 'BROOKLYN', 11230, 40.62179, -73.970024, 1.0, 0.0, 0, 0, 0, 0, 1, 0, 'Unspecified', 4675373, 2, 0, 1, 1, 'Other')
('2021-09-11', '9:35', 'BROOKLYN', 11208, 40.667202, -73.8665, 0.0, 0.0, 0, 0, 0, 0, 0, 0, 'Unspecified', 4456314, 5, 1, 9, 0, 'Car')


In [22]:
main_df.columns

Index(['CRASH DATE', 'CRASH TIME', 'BOROUGH', 'ZIP CODE', 'LATITUDE',
       'LONGITUDE', 'NUMBER OF PERSONS INJURED', 'NUMBER OF PERSONS KILLED',
       'NUMBER OF PEDESTRIANS INJURED', 'NUMBER OF PEDESTRIANS KILLED',
       'NUMBER OF CYCLIST INJURED', 'NUMBER OF CYCLIST KILLED',
       'NUMBER OF MOTORIST INJURED', 'NUMBER OF MOTORIST KILLED',
       'CONTRIBUTING FACTOR VEHICLE 1', 'COLLISION_ID', 'day_of_week',
       'is_weekend', 'hour', 'injury_flag', 'vehicle_category'],
      dtype='object')

In [24]:
pd.read_sql("""
SELECT *
FROM nyc_car_crash
LIMIT 5;
""", engine)

,CRASH DATE,CRASH TIME,BOROUGH,ZIP CODE,LATITUDE,LONGITUDE,NUMBER OF PERSONS INJURED,NUMBER OF PERSONS KILLED,NUMBER OF PEDESTRIANS INJURED,NUMBER OF PEDESTRIANS KILLED,...,NUMBER OF CYCLIST KILLED,NUMBER OF MOTORIST INJURED,NUMBER OF MOTORIST KILLED,CONTRIBUTING FACTOR VEHICLE 1,COLLISION_ID,day_of_week,is_weekend,hour,injury_flag,vehicle_category
0,2023-11-01,1:29,BROOKLYN,11230,40.621790,-73.970024,1.0,0.0,0,0,...,0,1,0,Unspecified,4675373,2,0,1,1,Other
1,2021-09-11,9:35,BROOKLYN,11208,40.667202,-73.866500,0.0,0.0,0,0,...,0,0,0,Unspecified,4456314,5,1,9,0,Car
2,2021-12-14,8:13,BROOKLYN,11233,40.683304,-73.917274,0.0,0.0,0,0,...,0,0,0,Unspecified,4486609,1,0,8,0,Other
3,2021-12-14,17:05,UNKNOWN,0,40.709183,-73.956825,0.0,0.0,0,0,...,0,0,0,Passing Too Closely,4486555,1,0,17,0,Car
4,2021-12-14,8:17,BRONX,10475,40.868160,-73.831480,2.0,0.0,0,0,...,0,2,0,Unspecified,4486660,1,0,8,1,Car


In [27]:
pd.read_sql("""
SELECT BOROUGH, COUNT(*) AS TOTAL_CRASH
FROM nyc_car_crash 
GROUP BY BOROUGH
ORDER BY TOTAL_CRASH DESC;
            """,engine)

,BOROUGH,TOTAL_CRASH
0,BROOKLYN,498993
1,UNKNOWN,481797
2,QUEENS,416524
3,MANHATTAN,343626
4,BRONX,230307
5,STATEN ISLAND,65150


In [29]:
## most dangerous borough
pd.read_sql("""SELECT BOROUGH ,SUM("NUMBER OF PERSONS KILLED") AS death
            FROM nyc_car_crash
            GROUP BY BOROUGH 
            ORDER BY death DESC;""",engine)

,BOROUGH,death
0,UNKNOWN,1046.0
1,BROOKLYN,728.0
2,QUEENS,600.0
3,MANHATTAN,390.0
4,BRONX,321.0
5,STATEN ISLAND,108.0


In [31]:
## INCIDENT PER HOUR
pd.read_sql("""SELECT hour,COUNT(*) AS crash_count
            FROM nyc_car_crash
            GROUP BY BOROUGH,hour
            ORDER BY crash_count DESC;""",engine)

,hour,crash_count
0,16,36360
1,17,35532
2,14,33678
3,16,33251
4,17,33159
...,...,...
139,1,843
140,2,741
141,5,713
142,3,655
